# **Load necessary libraries**

In [1]:
from transformers import BartTokenizer, BartForSequenceClassification, Trainer, TrainingArguments
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoModelForSeq2SeqLM
from sklearn.model_selection import train_test_split
import random
import nltk
from nltk.corpus import wordnet

In [2]:
# set the device to cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# **Loading labled data**

In [3]:
raw_train = pd.read_csv(r"https://raw.githubusercontent.com/MohammadWaleed339/bert-for-classification/refs/heads/master/labeled_traning_data.csv", index_col = False)

## a. Cleaning text

In [4]:
import re
def clean_text(text):
    if not isinstance(text, str):  # handle NaN or non-string
        return ""
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[^a-zA-Z0-9.,!?;:()\-\s]", "", text)  # keep basic chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
raw_train['article1'] = raw_train['article1'].apply(clean_text)
raw_train['article2'] = raw_train['article2'].apply(clean_text)

In [6]:
data_labled = raw_train

## b. Formate text for input in bert

In [7]:
data_labled['text'] = data_labled.apply(lambda row: f"[CLS] {row['article1']} [SEP] {row['article2']} [SEP]", axis=1)
data_labled = data_labled[['text', 'real_text_id']]
data_labled.columns = ['text', 'labels']

In [8]:
data_labled['labels'] = data_labled['labels'] - 1   #bcz target must be 0-1 not 1-2 so minus 1
data_labled

C:\Users\moham\AppData\Local\Temp\ipykernel_31904\1132998068.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_labled['labels'] = data_labled['labels'] - 1   #bcz target must be 0-1 not 1-2 so minus 1


,text,labels
0,[CLS] The VIRSA (Visible Infrared Survey Teles...,0
1,[CLS] China The goal of this project involves ...,1
2,[CLS] Scientists can learn about how galaxies ...,0
3,[CLS] China The study suggests that multiple s...,1
4,[CLS] Dinosaur Rex was excited about his new t...,1
...,...,...
90,[CLS] A main focus of modern cosmology is to u...,1
91,"[CLS] APEX, as its name suggests, serves as a ...",0
92,[CLS] FORS1 and FORS2 are early instruments of...,1
93,[CLS] The observations of the Pluto-Charon sys...,1


In [9]:
# Flip the dataset to increase the total training dataset
raw_train2 = pd.read_csv(r"https://raw.githubusercontent.com/MohammadWaleed339/bert-for-classification/refs/heads/master/labeled_traning_data.csv", index_col = False)

In [10]:
raw_train2['article1'] = raw_train2['article1'].apply(clean_text)
raw_train2['article2'] = raw_train2['article2'].apply(clean_text)

In [11]:
data_labled2 = raw_train2

In [12]:
data_labled2['text'] = data_labled2.apply(lambda row: f"[CLS] { row['article2']} [SEP] {row['article1']} [SEP]", axis=1)
data_labled2 = data_labled2.drop(columns = ['article1', 'article2', 'folder_id'])

In [13]:
# map labels to corresponding 0-1
data_labled2['labels'] = data_labled2['real_text_id'].map(lambda x: 0 if x == 2 else 1)
data_labled2 = data_labled2.drop(columns = ['real_text_id']) 
data_labled2.shape

(95, 2)

In [14]:
# add both data to increase training data.
data_labled = pd.concat([data_labled, data_labled2], axis = 0)
data_labled.index = range(190)

# **Import the actual test set with no labels**

In [75]:
url = "https://raw.githubusercontent.com/MohammadWaleed339/bert-for-classification/refs/heads/master/test_article_pairs.csv"
test_unlabled = pd.read_csv(url, encoding = 'latin-1')
test_unlabled.columns = ['folder_id','article1', 'article2']
test_unlabled = pd.DataFrame(test_unlabled)

In [76]:
test_unlabled = test_unlabled.drop(columns = ['folder_id'])
test_unlabled.index = range(1067)

In [77]:
test_unlabled['text'] = test_unlabled.apply(lambda row: f"[CLS] {row['article1']} [SEP] {row['article2']} [SEP]", axis = 1)
test_unlabled = test_unlabled.drop(columns=['article1','article2'])

In [78]:
test_unlabled

,text
0,[CLS] underground exploration on SN's birth ha...
1,[CLS] This research aimed to understand how st...
2,[CLS] Using OmegaCAM's wide field capabilities...
3,[CLS] AssemblyCulture AssemblyCulture Assembly...
4,[CLS] Research indicates that spiral and ellip...
...,...
1062,[CLS] Alongside the detailed studies mentioned...
1063,"[CLS] At this meeting, we gained a new outlook..."
1064,[CLS] ESO Reflex is designed to handle essenti...
1065,[CLS] Even greater angular resolution is possi...


# Trying data augmentation

In [18]:
import random
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

In [19]:

PROTECTED_TOKENS = {"[CLS]", "[SEP]"}

def get_synonym(word):
    """Get a random synonym for a given word using WordNet."""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name().lower() != word.lower():  # avoid same word
                synonyms.add(lemma.name().replace("_", " "))
    if synonyms:
        return random.choice(list(synonyms))
    return None

def synonym_replacement(text, n=20):
    """Randomly replace up to n words in the text with synonyms."""
    words = word_tokenize(text)
    candidate_indices = [i for i, w in enumerate(words) 
                         if w.isalpha() and w not in PROTECTED_TOKENS
                        ]  # only words, no punctuation

    if not candidate_indices:
        return text

    num_to_replace = min(n, len(candidate_indices))
    indices_to_replace = random.sample(candidate_indices, num_to_replace)

    new_words = words[:]
    for idx in indices_to_replace:
        synonym = get_synonym(words[idx])
        if synonym:  # only replace if synonym found
            new_words[idx] = synonym

    return " ".join(new_words)

In [20]:
df3 = pd.DataFrame()

In [21]:
df3['text'] = data_labled['text'].apply(synonym_replacement)
df3['labels'] = data_labled['labels'].astype(int)
data_labled = pd.concat([data_labled, df3], axis = 0)
data_labled.index = range(380)

# Split the data_labled

In [22]:
# Split into train/test
train_df, val_df = train_test_split(data_labled, test_size=0.3, random_state = 1)

In [23]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# **(1)Using sciBert** 

In [24]:
# Load SciBERT
MODEL_NAME = "allenai/scibert_scivocab_uncased"
scibert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_scibert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
# Tokenization function
def tokenize_function(example):
    # Tokenizer will handle [SEP] inside the text automatically
    return scibert_tokenizer(example["text"], truncation=True, padding="max_length", max_length = 512)

scibert_train_dataset = train_dataset.map(tokenize_function, batched=True)
scibert_val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/266 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [26]:
# Set format for PyTorch
scibert_train_ds = scibert_train_dataset.remove_columns(["text", "__index_level_0__", "token_type_ids"])
scibert_val_ds = scibert_val_dataset.remove_columns(["text", "__index_level_0__", "token_type_ids"])

In [27]:
scibert_train_ds.set_format("torch")
scibert_val_ds.set_format("torch")

In [28]:
scibert_train_ds

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 266
})

In [29]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./scibert-classifier",
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3.5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50
)
# Trainer
trainer_scibert = Trainer(
    model=model_scibert,
    args=training_args,
    train_dataset=scibert_train_ds,
    eval_dataset=scibert_val_ds
)

In [30]:
# Train model
trainer_scibert.train()

wandb: Currently logged in as: mohammadwaleed339 (mohammadwaleed339-aligarh-muslim-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,0.664000


TrainOutput(global_step=68, training_loss=0.6290839139152976, metrics={'train_runtime': 61.3487, 'train_samples_per_second': 8.672, 'train_steps_per_second': 1.108, 'total_flos': 139975081451520.0, 'train_loss': 0.6290839139152976, 'epoch': 2.0})

In [31]:
# Train model
trainer_scibert.train()

Step,Training Loss
50,0.414900


TrainOutput(global_step=68, training_loss=0.37054913885453167, metrics={'train_runtime': 52.7423, 'train_samples_per_second': 10.087, 'train_steps_per_second': 1.289, 'total_flos': 139975081451520.0, 'train_loss': 0.37054913885453167, 'epoch': 2.0})

### ⬆️running above trainer twice increases the accuracy above 90% but doubling epoch does not.⬆️

In [32]:
from tabulate import tabulate
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
def evaluate_scibert(model, tokenizer, texts, labels, max_length=512, device=None):
    """
    Predicts labels with SciBERT and computes accuracy.
    
    Args:
        model: Trained SciBERT model (AutoModelForSequenceClassification).
        tokenizer: SciBERT tokenizer.
        texts: List of strings (each string = "text1 [SEP] text2").
        labels: List or tensor of true labels (0/1).
        max_length: Max token length (default=512).
        device: "cuda" or "cpu".
        
    Returns:
        accuracy: float, prediction accuracy
        preds: list of predicted labels
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.to(device)
    model.eval()

    preds = []
    with torch.no_grad():
        for text in texts:
            # Tokenize
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding="max_length",
                max_length=max_length
            )
            # Move to device
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Forward pass
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=-1).cpu().item()
            
            preds.append(pred)  
    # Calculate accuracy
    acc = accuracy_score(labels, preds)
    if isinstance(labels, torch.Tensor):
        labels = list(labels)

    correct = sum(v == u for v, u in zip(labels, preds))
    n_total = len(labels)
    report = classification_report(labels, preds, output_dict = True)
    report_df = pd.DataFrame(report).round(3).T
    print(tabulate(report_df, headers='keys', tablefmt='pretty', floatfmt=".2f"))
    print(f"out of {n_total} total {correct} are correct")
    return acc
    

In [33]:
train_df, val_df = train_test_split(data_labled, test_size=0.3, random_state = 199)

In [34]:
# test accuracy
evaluate_scibert(model_scibert, scibert_tokenizer, val_df['text'], val_df['labels'], max_length=512, device=None)

+--------------+-----------+--------+----------+---------+
|              | precision | recall | f1-score | support |
+--------------+-----------+--------+----------+---------+
|      0       |   0.807   | 0.939  |  0.868   |  49.0   |
|      1       |   0.947   | 0.831  |  0.885   |  65.0   |
|   accuracy   |   0.877   | 0.877  |  0.877   |  0.877  |
|  macro avg   |   0.877   | 0.885  |  0.877   |  114.0  |
| weighted avg |   0.887   | 0.877  |  0.878   |  114.0  |
+--------------+-----------+--------+----------+---------+
out of 114 total 100 are correct


0.8771929824561403

In [35]:
# evaluate on train data to see if the models is over fitting.
evaluate_scibert(model_scibert, scibert_tokenizer, train_df['text'], train_df['labels'], max_length=512, device=None)

+--------------+-----------+--------+----------+---------+
|              | precision | recall | f1-score | support |
+--------------+-----------+--------+----------+---------+
|      0       |   0.895   | 0.965  |  0.928   |  141.0  |
|      1       |   0.956   | 0.872  |  0.912   |  125.0  |
|   accuracy   |   0.921   | 0.921  |  0.921   |  0.921  |
|  macro avg   |   0.925   | 0.918  |   0.92   |  266.0  |
| weighted avg |   0.924   | 0.921  |  0.921   |  266.0  |
+--------------+-----------+--------+----------+---------+
out of 266 total 245 are correct


0.9210526315789473

### *⬆️ if accuracy above is less than 90% run the trainer again ⬆️*
- running above trainer twice increases the accuracy above 90% but doubling epoch does not.

## Check the preds values and probability

In [36]:
from torch.nn import functional as F
def prediction_func(texts, certainity = 0.95):
    preds = []
    class0 = []
    class1 = []
    probability = []
    texts = texts
    threshold = 0.5  # set your threshold here
    choosen_text = []
    model_scibert.to(device)
    model_scibert.eval()
    
    with torch.no_grad():   # don't track gradient to save memory and increase speed.
        for text in texts:
            inputs = scibert_tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding="max_length",
                max_length=512
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
    
            outputs = model_scibert(**inputs)
    
            # Get probability for class 0-1
            probs = F.softmax(outputs.logits, dim=-1)
            probability.append(probs)
    
            prob_class1 = probs[:, 1].cpu().item()
            if prob_class1 >= certainity :
                class1.append(prob_class1)
                pred = 1 if prob_class1 >= threshold else 0
                choosen_text.append([text, pred, prob_class1])
    
            prob_class0 = probs[:, 0].cpu().item()
            if prob_class0 >= certainity :
                class0.append(prob_class0)
                pred = 1 if prob_class1 >= threshold else 0
                choosen_text.append([text, pred, round(prob_class0, 2)])
        choosen_text = pd.DataFrame(choosen_text)
        return choosen_text

In [37]:
choosen_text = prediction_func(test_unlabled['text'][50:300], certainity = 0.95)

In [38]:
choosen_text.columns = ['text', 'labels', 'probability']

In [39]:
test_text = choosen_text[['text', 'labels']]

In [40]:
data_labled = pd.concat([test_text, data_labled], axis = 0)

In [41]:
data_labled = data_labled.reset_index(drop=True)

In [42]:
data_labled.shape

(460, 2)

## **Training on final data**
### ***How final data is made***
- We swaped original data and labels and added into training data.
- Replaced few words with synonym and added into training data.
- Took top predictions (which model is >95% sure about) based on above train data
- Added those top prediction in the train data and made final training data
### final_data has original cleaned labled data, same data with some variation and ~150 unlabled test data making it ideal for model to capture every possible details.

In [43]:
final_data = data_labled

In [44]:
final_data.index = range(data_labled.shape[0])

In [45]:
## save file to csv
# final_data.to_csv("kaggle-final-train-dataset.csv")

In [46]:
## Save only the trained weights of your model
# torch.save(model_scibert.state_dict(), "model_scibert.pth")

## **Final text processing and training** 

In [47]:
# comment this line to use current data, this is what i got after prediction and concatenation.
final_data = pd.read_csv(r"C:\\Users\\moham\\Jupyter_files\\kaggle-final-train-dataset.csv", index_col=0)

In [48]:
# Split into train/test
final_train_df, final_val_df = train_test_split(final_data, test_size=0.3, random_state = 3)

In [49]:
final_train_df.index = range(final_train_df.shape[0])

In [50]:
final_val_df.index = range(final_val_df.shape[0])

In [51]:
final_val_df.index = range(final_val_df.shape[0])

In [52]:
final_train_df = Dataset.from_pandas(final_train_df)
final_val_df = Dataset.from_pandas(final_val_df)

In [53]:
final_train_dataset = final_train_df.map(tokenize_function, batched=True)
final_val_dataset = final_val_df.map(tokenize_function, batched=True)

Map:   0%|          | 0/367 [00:00<?, ? examples/s]

Map:   0%|          | 0/158 [00:00<?, ? examples/s]

In [54]:
final_train_tokd = final_train_dataset.remove_columns(["text", "token_type_ids"])
final_val_tokd = final_val_dataset.remove_columns(["text", "token_type_ids"])

In [55]:
final_train_tokd.set_format("torch")
final_val_tokd.set_format("torch")

In [56]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./scibert-classifier",
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3.5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50
)
# Trainer
trainer_scibert = Trainer(
    model=model_scibert,
    args=training_args,
    train_dataset=final_train_tokd,
    eval_dataset=final_val_df
)

In [57]:
# Train model
trainer_scibert.train()

Step,Training Loss
50,0.307400


TrainOutput(global_step=92, training_loss=0.25089669745901355, metrics={'train_runtime': 71.3883, 'train_samples_per_second': 10.282, 'train_steps_per_second': 1.289, 'total_flos': 193123514634240.0, 'train_loss': 0.25089669745901355, 'epoch': 2.0})

In [58]:
# test accuracy
evaluate_scibert(model_scibert, scibert_tokenizer, final_val_df['text'], final_val_df['labels'], max_length=512, device=None)

+--------------+-----------+--------+----------+---------+
|              | precision | recall | f1-score | support |
+--------------+-----------+--------+----------+---------+
|      0       |   0.893   | 0.948  |   0.92   |  97.0   |
|      1       |   0.909   |  0.82  |  0.862   |  61.0   |
|   accuracy   |   0.899   | 0.899  |  0.899   |  0.899  |
|  macro avg   |   0.901   | 0.884  |  0.891   |  158.0  |
| weighted avg |   0.899   | 0.899  |  0.898   |  158.0  |
+--------------+-----------+--------+----------+---------+
out of 158 total 142 are correct


0.8987341772151899

In [59]:
text_class = prediction_func(test_unlabled['text'], certainity = 0.5)

In [60]:
text_class['id'] = range(text_class.shape[0])

In [61]:
text_class.columns = ['text', 'labels', 'probability', 'id']

In [62]:
text_class['labels'] = text_class['labels']+1

In [63]:
text_class[['id','labels']].to_csv('Final_prediction.csv', index = False)

In [64]:
text_class

,text,labels,probability,id
0,[CLS] underground exploration on SN's birth ha...,1,0.990000,0
1,[CLS] This research aimed to understand how st...,1,0.990000,1
2,[CLS] Using OmegaCAM's wide field capabilities...,1,0.940000,2
3,[CLS] AssemblyCulture AssemblyCulture Assembly...,2,0.995352,3
4,[CLS] Research indicates that spiral and ellip...,1,0.990000,4
...,...,...,...,...
1062,[CLS] Alongside the detailed studies mentioned...,1,0.950000,1062
1063,"[CLS] At this meeting, we gained a new outlook...",1,0.980000,1063
1064,[CLS] ESO Reflex is designed to handle essenti...,1,0.980000,1064
1065,[CLS] Even greater angular resolution is possi...,2,0.993332,1065
